
# BCG unsuitable-segment estimation using wavelets

This notebook implements the core segmentation idea from the paper:

**“An Improved Estimation of Unsuitable Segments of Ballistocardiography Records Using Wavelet Transforms.”**

It includes:

- bandpass filtering for BCG and optional ECG
- continuous wavelet transform (CWT) of the BCG
- the proposed segmentation method using:
  - moving standard deviation of the raw BCG
  - moving standard deviation of the CWT
- a baseline method using variance of the raw BCG only
- optional coverage factor (CF) estimation using ECG beat counts
- plots for visual comparison

## Important notes

This notebook is a practical re-implementation based on the paper description. A few details are not fully specified in the paper, so the implementation makes reasonable engineering choices:

- the paper states a 1-second moving window
- the paper uses the **Gaus2** mother wavelet at **scale 16**
- the paper defines:
  - `Ws = (W1 + W2)/2`
  - `Thd_max = mean(W1 + W2)`
  - `Thd_min = mean(W1 + W2) * 0.1`
- the paper expands each threshold crossing by **±150 ms**

You may need to adapt:
- file loading
- sensor channel selection
- ECG peak detection settings
- exact plotting ranges


In [12]:

import numpy as np
import sys
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
import pywt
from pathlib import Path

#  & "C:\Users\Christian Damsgaard\AppData\Local\Programs\Python\Python310\python.exe" Use this for path for pip install 
plt.rcParams["figure.figsize"] = (14, 5)



## 1. Helper functions


In [ ]:

def bandpass_filter(x, fs, lowcut, highcut, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype="band")
    return signal.filtfilt(b, a, x)


def moving_std(x, window_samples):
    # centered moving std using pandas for convenience
    s = pd.Series(np.asarray(x))
    return s.rolling(window=window_samples, center=True, min_periods=1).std(ddof=0).to_numpy()


def moving_var(x, window_samples):
    s = pd.Series(np.asarray(x))
    return s.rolling(window=window_samples, center=True, min_periods=1).var(ddof=0).to_numpy()


def expand_mask(mask, radius_samples):
    mask = np.asarray(mask, dtype=bool)
    kernel = np.ones(2 * radius_samples + 1, dtype=int)
    expanded = np.convolve(mask.astype(int), kernel, mode="same") > 0
    return expanded


def cwt_gaus2_scale16(x):
    # PyWavelets continuous wavelet transform
    scales = [16]
    coeffs, freqs = pywt.cwt(x, scales=scales, wavelet="gaus2")
    # coeffs shape: (num_scales, n_samples)
    return coeffs[0]


def segmentation_proposed(bcg, fs):
    window_samples = int(fs * 1.0)        # 1 second
    margin_samples = int(fs * 0.150)      # ±150 ms

    cwt_bcg = cwt_gaus2_scale16(bcg)

    W1 = moving_std(bcg, window_samples)
    W2 = moving_std(cwt_bcg, window_samples)

    Ws = (W1 + W2) / 2.0

    thd_max = np.mean(W1 + W2)
    thd_min = np.mean(W1 + W2) * 0.1

    high_artifact = Ws > thd_max
    low_signal = Ws < thd_min

    high_artifact_expanded = expand_mask(high_artifact, margin_samples)
    low_signal_expanded = expand_mask(low_signal, margin_samples)

    # segmentation labels from the paper:
    # 2 = above max threshold (artifact / saturation / large variation)
    # 1 = between thresholds (usable)
    # 0 = below min threshold (weak or absent signal)
    seg = np.ones_like(Ws, dtype=int)
    seg[low_signal_expanded] = 0
    seg[high_artifact_expanded] = 2

    return {
        "cwt_bcg": cwt_bcg,
        "W1": W1,
        "W2": W2,
        "Ws": Ws,
        "thd_max": thd_max,
        "thd_min": thd_min,
        "seg": seg,
        "usable_mask": seg == 1,
        "artifact_mask": seg == 2,
        "low_signal_mask": seg == 0,
    }


def segmentation_variance_baseline(bcg, fs):
    window_samples = int(fs * 1.0)
    margin_samples = int(fs * 0.150)

    Wvar = moving_var(bcg, window_samples)

    # The paper compares to a more traditional variance-window approach.
    # It does not fully specify all threshold details for the baseline in the extracted text,
    # so we use an analogous threshold structure for a fair engineering comparison.
    thd_max = np.mean(Wvar)
    thd_min = np.mean(Wvar) * 0.1

    high_artifact = Wvar > thd_max
    low_signal = Wvar < thd_min

    high_artifact_expanded = expand_mask(high_artifact, margin_samples)
    low_signal_expanded = expand_mask(low_signal, margin_samples)

    seg = np.ones_like(Wvar, dtype=int)
    seg[low_signal_expanded] = 0
    seg[high_artifact_expanded] = 2

    return {
        "Wvar": Wvar,
        "thd_max": thd_max,
        "thd_min": thd_min,
        "seg": seg,
        "usable_mask": seg == 1,
        "artifact_mask": seg == 2,
        "low_signal_mask": seg == 0,
    }


def detect_ecg_rpeaks(ecg, fs):
    # Simple ECG peak detector; tune as needed for your data
    # Distance ~ 0.4 s prevents unrealistically close peaks
    distance = int(0.4 * fs)
    prominence = np.std(ecg) * 0.5
    peaks, props = signal.find_peaks(ecg, distance=distance, prominence=prominence)
    return peaks


def coverage_factor_from_ecg(seg_usable_mask, rpeaks):
    if len(rpeaks) == 0:
        return np.nan, 0, 0
    usable_beats = int(np.sum(seg_usable_mask[rpeaks]))
    total_beats = len(rpeaks)
    cf = 100.0 * usable_beats / total_beats
    return cf, usable_beats, total_beats



## 2. Load your data

This cell expects at least a BCG signal.

You can load:
- a CSV with columns like `time`, `bcg`, `ecg`
- or replace this with your own dataset loader

### Example expected columns
- `bcg`
- optional `ecg`
- optional `time`

Edit `DATA_PATH` and the column names below.


In [ ]:

DATA_PATH = Path("your_data.csv")   # <-- change this
FS = 1000                          # paper uses 1 kHz

BCG_COLUMN = "bcg"
ECG_COLUMN = "ecg"   # set to None if not available

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    bcg_raw = df[BCG_COLUMN].to_numpy(dtype=float)
    ecg_raw = df[ECG_COLUMN].to_numpy(dtype=float) if ECG_COLUMN and ECG_COLUMN in df.columns else None
    time = np.arange(len(bcg_raw)) / FS
    print(f"Loaded {DATA_PATH.name} with {len(bcg_raw)} samples.")
else:
    print("No data file found yet. The demo cell below can generate synthetic data so the notebook still runs.")
    bcg_raw = None
    ecg_raw = None
    time = None



## 3. Optional synthetic demo data

This is only for testing the notebook structure.

It creates:
- a heartbeat-like BCG signal
- respiration-like baseline modulation
- a motion artifact burst
- optional ECG-like reference peaks

Skip this section if you already loaded real data.


In [ ]:

def make_synthetic_demo(fs=1000, duration_s=40):
    t = np.arange(0, duration_s, 1/fs)

    heart_hz = 1.2
    resp_hz = 0.25

    # crude BCG-like content
    bcg = 0.4 * np.sin(2*np.pi*heart_hz*t)
    bcg += 0.15 * np.sin(2*np.pi*(2*heart_hz)*t)
    bcg *= (1.0 + 0.4*np.sin(2*np.pi*resp_hz*t))  # amplitude modulation
    bcg += 0.08 * np.sin(2*np.pi*0.05*t)          # slow drift
    bcg += 0.03 * np.random.randn(len(t))         # noise

    # motion artifact burst
    artifact_region = (t > 18) & (t < 21)
    bcg[artifact_region] += 1.0 * np.random.randn(np.sum(artifact_region))

    # weak signal / missing signal
    low_region = (t > 30) & (t < 33)
    bcg[low_region] *= 0.05

    # ECG-like spikes
    rr = int(fs / heart_hz)
    rpeaks = np.arange(rr, len(t), rr)
    ecg = 0.02 * np.random.randn(len(t))
    for rp in rpeaks:
        if rp < len(ecg):
            ecg[rp:rp+3] += 1.0

    return t, bcg, ecg

if bcg_raw is None:
    time, bcg_raw, ecg_raw = make_synthetic_demo(fs=FS, duration_s=40)
    print("Using synthetic demo data.")



## 4. Preprocess signals

According to the paper:
- BCG bandwidth: `0.3–24 Hz`
- ECG bandwidth: `0.5–40 Hz`


In [ ]:

bcg = bandpass_filter(bcg_raw, FS, 0.3, 24.0, order=4)

if ecg_raw is not None:
    ecg = bandpass_filter(ecg_raw, FS, 0.5, 40.0, order=4)
else:
    ecg = None

print("Preprocessing complete.")



## 5. Run both segmentation methods


In [ ]:

res_prop = segmentation_proposed(bcg, FS)
res_var = segmentation_variance_baseline(bcg, FS)

print("Proposed method thresholds:")
print("  thd_max =", res_prop["thd_max"])
print("  thd_min =", res_prop["thd_min"])

print("\nVariance baseline thresholds:")
print("  thd_max =", res_var["thd_max"])
print("  thd_min =", res_var["thd_min"])



## 6. Plot full-record overview


In [ ]:

fig, axes = plt.subplots(4, 1, figsize=(15, 10), sharex=True)

axes[0].plot(time, bcg, label="BCG")
axes[0].set_title("Filtered BCG")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(time, res_prop["cwt_bcg"], label="CWT(BCG), gaus2 scale 16")
axes[1].set_title("Wavelet coefficient")
axes[1].legend()
axes[1].grid(True)

axes[2].plot(time, res_prop["Ws"], label="Ws = (W1 + W2)/2")
axes[2].axhline(res_prop["thd_max"], linestyle="--", label="Thd max")
axes[2].axhline(res_prop["thd_min"], linestyle="--", label="Thd min")
axes[2].set_title("Proposed method metric")
axes[2].legend()
axes[2].grid(True)

axes[3].plot(time, res_var["Wvar"], label="Wvar")
axes[3].axhline(res_var["thd_max"], linestyle="--", label="Thd max")
axes[3].axhline(res_var["thd_min"], linestyle="--", label="Thd min")
axes[3].set_title("Variance-only baseline metric")
axes[3].legend()
axes[3].grid(True)

axes[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()



## 7. Plot segmentation labels

Label meaning:
- `0` = weak / absent signal
- `1` = usable
- `2` = artifact / large variation


In [ ]:

plt.figure(figsize=(15, 6))
plt.plot(time, bcg, label="BCG", alpha=0.8)
plt.plot(time, res_prop["seg"], label="Seg proposed", linewidth=2)
plt.plot(time, res_var["seg"], label="Seg variance", linewidth=2, alpha=0.8)
if ecg is not None:
    plt.plot(time, ecg, label="ECG", alpha=0.7)
plt.title("Signal and segmentation comparison")
plt.xlabel("Time (s)")
plt.legend()
plt.grid(True)
plt.show()



## 8. Zoom into a time interval

Change `t0` and `t1` to inspect examples like the figures in the paper.


In [ ]:

t0, t1 = 15, 23
idx = (time >= t0) & (time <= t1)

plt.figure(figsize=(15, 7))
plt.plot(time[idx], bcg[idx], label="BCG")
plt.plot(time[idx], res_prop["Ws"][idx], label="Ws")
plt.plot(time[idx], res_var["Wvar"][idx], label="Wvar")
plt.plot(time[idx], res_prop["seg"][idx], label="Seg proposed")
plt.plot(time[idx], res_var["seg"][idx], label="Seg variance")
plt.axhline(res_prop["thd_max"], linestyle="--", label="Prop Thd max")
plt.axhline(res_prop["thd_min"], linestyle="--", label="Prop Thd min")
plt.title(f"Zoomed view: {t0} s to {t1} s")
plt.xlabel("Time (s)")
plt.legend()
plt.grid(True)
plt.show()



## 9. Coverage factor (CF) using ECG beat counts

The paper defines coverage factor as:

`CF = suitable heartbeats / total heartbeats × 100`

This cell uses ECG R-peaks as the reference heartbeat count.


In [ ]:

if ecg is not None:
    rpeaks = detect_ecg_rpeaks(ecg, FS)

    cf_prop, usable_prop, total_beats = coverage_factor_from_ecg(res_prop["usable_mask"], rpeaks)
    cf_var, usable_var, _ = coverage_factor_from_ecg(res_var["usable_mask"], rpeaks)

    print(f"Total beats: {total_beats}")
    print(f"Proposed method: CF = {cf_prop:.2f}% ({usable_prop}/{total_beats})")
    print(f"Variance baseline: CF = {cf_var:.2f}% ({usable_var}/{total_beats})")
    print(f"Difference in beats kept: {usable_prop - usable_var}")
else:
    print("No ECG available, so CF cannot be computed in this cell.")



## 10. Optional: save segmentation results


In [ ]:

out = pd.DataFrame({
    "time_s": time,
    "bcg_filtered": bcg,
    "cwt_bcg": res_prop["cwt_bcg"],
    "W1_std_bcg": res_prop["W1"],
    "W2_std_cwt": res_prop["W2"],
    "Ws": res_prop["Ws"],
    "seg_proposed": res_prop["seg"],
    "Wvar": res_var["Wvar"],
    "seg_variance": res_var["seg"],
})

if ecg is not None:
    out["ecg_filtered"] = ecg

OUT_PATH = Path("bcg_segmentation_results.csv")
out.to_csv(OUT_PATH, index=False)
print(f"Saved results to {OUT_PATH.resolve()}")



## 11. How to adapt this notebook to your own project

### If your data comes from a piezo / Arduino / ESP32 setup
You will likely need to:
- replace the CSV loader with your acquisition format
- verify the sample rate
- tune the BCG bandpass
- decide whether you want:
  - online detection
  - or offline post-processing

### If you want to match the paper more closely
You may want to:
- use the same bed-based dataset
- use the exact same channel selection
- verify whether the authors used absolute CWT coefficients or signed coefficients
- inspect whether threshold handling at the edges should be adjusted
- tune ECG peak detection to match their reference counting

### If you want a more robust comparison
You can add:
- precision / recall for artifact regions, if you have labels
- beat detection performance before and after segmentation
- per-record summary tables like the paper's Table 2
